# Phase 6: build the files for the online demo

The demo is a small web app (Streamlit, `demo/streamlit_app.py` in the repository): pick an example or upload a
mammogram and see the model's malignancy score, density grade and heatmaps. This notebook prepares the two things
the app needs and that are not in the repository yet:

- `model.pt`: the **paper-design model from Phase 4** (no retraining), with its calibration;
- `examples/`: 6 **randomly drawn** official-test images (3 malignant, 3 benign), saved exactly as the model sees them.

It then checks that the app reproduces the Phase 5 predictions and runs the app once in test mode.

**Before running**
1. Settings → **Internet on**. Accelerator: **None (CPU) is enough**.
2. Add Input → **"CBIS-DDSM: Breast Cancer Image Dataset"** (by *awsaf49*).
3. Add Input → **Your Work → Notebooks → `notebookb86c6df37c`** (the Phase 4 notebook with `checkpoints/`).
4. Run all cells (about 10 minutes).
5. From **Output** (right panel, `/kaggle/working`) download **`demo_bundle.zip`** and **`demo_results.zip`**, and
   put both into your project folder (`Documents\mammography-multitask-ai`). Claude takes it from there.


In [ ]:
# 1) Get the code, find the Phase 4 checkpoint
import os, subprocess, sys
REPO = "/kaggle/working/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sweetmAGIciaN7/mammography-multitask-ai.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Could not download the code. Is Internet ON (Settings -> Internet)?\n" + r.stderr)
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
os.environ["PYTHONPATH"] = f"{REPO}/src"
sys.path.insert(0, f"{REPO}/src")
from mammo.experiments.attention import find_checkpoints
ck = find_checkpoints("auto").get("mt_cbam")
print("paper-design checkpoint:", ck or "MISSING -> add the Phase 4 notebook (notebookb86c6df37c) as an input")


In [ ]:
# 2) Tests (includes an end-to-end build with a tiny untrained model). Must end with "passed".
!cd /kaggle/working/repo && python -m pytest -q tests/test_demo.py 2>&1 | tail -3


In [ ]:
# 3) Build model.pt + 6 random test images; check the demo reproduces the Phase 5 predictions exactly
!cd /kaggle/working/repo && python -m mammo.experiments.demo_export build
from IPython.display import Image, display
display(Image("/kaggle/working/results/demo/demo_preview.png"))


In [ ]:
# 4) Run the real web app once in Streamlit's test mode (no browser needed)
!pip install -q "streamlit>=1.40" 2>&1 | tail -1
import shutil
shutil.copytree("/kaggle/working/demo", f"{REPO}/demo", dirs_exist_ok=True)
from streamlit.testing.v1 import AppTest
at = AppTest.from_file(f"{REPO}/demo/streamlit_app.py", default_timeout=300).run()
assert not at.exception, at.exception
print("app OK:", [m.label + " = " + m.value for m in at.metric])


In [ ]:
# 5) Pack the check files for Claude. Download BOTH zips from Output (/kaggle/working).
!cd /kaggle/working && rm -f demo_results.zip && zip -qr demo_results.zip results/demo && ls -lh demo_bundle.zip demo_results.zip
